In [ ]:
import pandas as pd
import tensorflow
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Input,Embedding,LSTM
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
import kagglehub

**importing dataset form kaggle**

In [ ]:
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")
print("Path to dataset files:", path)

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.
Path to dataset files: /kaggle/input/imdb-dataset-of-50k-movie-reviews


In [ ]:
df = pd.read_csv('/kaggle/input/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv')

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [ ]:
df['sentiment'].replace(to_replace=["positive","negative"],value=[1,0],inplace=True)

/tmp/ipython-input-167754365.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['sentiment'].replace(to_replace=["positive","negative"],value=[1,0],inplace=True)
/tmp/ipython-input-167754365.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['sentiment'].replace(to_replace=["positive","negat

In [ ]:
X = df['review'].to_numpy()
Y = df['sentiment'].to_numpy()

In [ ]:
x_train,x_test,y_train,y_test = train_test_split(X,Y,test_size=0.2)

In [ ]:
print(f"Train and Test datasets :\nTrain : {x_train.shape,y_train.shape},\nTest : {x_test.shape,y_test.shape} ")

Train and Test datasets :
Train : ((40000,), (40000,)),
Test : ((10000,), (10000,)) 


**Data preprocessing**

In [ ]:
# Tokenizing the input
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(x_train)
x_train = pad_sequences(tokenizer.texts_to_sequences(x_train),maxlen=200,padding ='post')
x_test = pad_sequences(tokenizer.texts_to_sequences(x_test),maxlen=200,padding ='post')


*Converting Text data into numeric values*

In [ ]:
model = Sequential()
model.add(Embedding(input_dim=5000,output_dim=128,input_length=200))
model.add(LSTM(128,dropout=0.2,recurrent_dropout=0.2))
model.add(Dense(1,activation="sigmoid"))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
model.compile(
    optimizer = "adam",
    loss = "binary_crossentropy",
    metrics=['accuracy']
)

In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.fit(
    x = x_train,
    y = y_train,
    epochs=5,
    validation_data = [x_test,y_test]
)

Epoch 1/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 488s 384ms/step - accuracy: 0.5501 - loss: 0.6818 - val_accuracy: 0.7825 - val_loss: 0.5208
Epoch 2/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 473s 379ms/step - accuracy: 0.8402 - loss: 0.3835 - val_accuracy: 0.8930 - val_loss: 0.2607
Epoch 3/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 483s 386ms/step - accuracy: 0.9121 - loss: 0.2245 - val_accuracy: 0.8997 - val_loss: 0.2496
Epoch 4/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 484s 387ms/step - accuracy: 0.9328 - loss: 0.1860 - val_accuracy: 0.8991 - val_loss: 0.2633
Epoch 5/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 489s 391ms/step - accuracy: 0.9473 - loss: 0.1478 - val_accuracy: 0.8912 - val_loss: 0.2871


In [ ]:
def Inference(text):
  input_text = pad_sequences(tokenizer.texts_to_sequences(text),maxlen=200,padding='post')
  pred = model.predict(input_text)
  if pred[0][0] > 0.5:
    print("positive Sentiment")
  else:
    print('Negative Sentiment')

In [ ]:
text = input()
pred = Inference(text)